### **Creating silver table**
### # will check for the latest timestamp and store it in a variable
# perform cleansing and basic operations

In [0]:
from pyspark.sql.functions import max

try:
    last_ts = spark.table("db_test.bronze.movies") \
        .select(max("ingestion_date")) \
        .collect()[0][0]
except:
    last_ts = None

print(last_ts)

In [0]:
# if last_ts:
#     bronze_df = bronze_df.filter(f"ingested_on > '{last_ts}'")
# we are not checking latest time here as now only we are creating our table
# for incremental load we can use if/else to read only the latest ingested date
bronze_df = spark.table("db_test.bronze.movies")
display(bronze_df)

In [0]:
from pyspark.sql.functions import col, when, lit

df = bronze_df \
    .withColumn("studio", when(col("studio").isNull(), lit("Not Available")).otherwise(col("studio"))) \
    .withColumn("imdb_rating", when(col("imdb_rating") == "NULL", lit("0")).otherwise(col("imdb_rating")))

In [0]:
df1 = df \
    .withColumn("release_year", col("release_year").cast("int")) \
    .withColumn("imdb_rating", col("imdb_rating").cast("decimal(10,1)")) \
    .withColumn("budget", col("budget").cast("double")) \
    .withColumn("revenue", col("revenue").cast("double"))

In [0]:
from pyspark.sql.functions import current_timestamp

df2 = df1 \
    .withColumn("ingested_on", current_timestamp()) \
    .withColumn("source", lit("db_test.bronze.movies"))
display(df2)

In [0]:
from delta.tables import DeltaTable


if not spark.catalog.tableExists("db_test.silver.movies"):
    df.write.format("delta").mode("overwrite").saveAsTable("db_test.silver.movies")
else:
    delta_table = DeltaTable.forName(spark, "db_test.silver.movies")
    delta_table.alias("target").merge(
    df.alias("source"),
    "target.title = source.title"
).whenMatchedUpdate(set={
    "industry": "source.industry",
    "release_year": "source.release_year",
    "imdb_rating": "source.imdb_rating",
    "studio": "source.studio"
}) \
.whenNotMatchedInsert(values={
    "title": "source.title",
    "industry": "source.industry",
    "release_year": "source.release_year",
    "imdb_rating": "source.imdb_rating",
    "studio": "source.studio",
    "budget": "source.budget",
    "revenue": "source.revenue",
    "unit": "source.unit",
    "currency": "source.currency",
    "language": "source.language",
    "ingested_on": "source.ingested_on",
    "source": "source.source"
}) \
.execute()

# Silver Table created